## 第16章　条件概率与贝叶斯定理 —— 已知信息改变判断

> 本章目标：理解条件概率 P(A|B)——"已知 B 发生了，A 的概率是多少"。掌握贝叶斯定理并亲手验证那个著名的反直觉结果：检测准确率 99%、发病率 1%，阳性后真实患病率仅约 50%。最后用朴素贝叶斯分类器手写一个垃圾短信识别器——20 行代码达到 90%+ 准确率。
> 前置知识：第 15 章（概率分布）、第 1 章（NumPy）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('env ready')



### 16.1　条件概率 —— 已知信息如何"缩小可能性空间"

掷一个公平骰子，P(点数为 6) = 1/6。但如果我告诉你"点数是偶数"——你的判断立刻收窄：现在只有 {2, 4, 6} 三种可能，P(点数为 6 | 已知是偶数) = 1/3。这就是**条件概率**的核心直觉：**新信息缩小了样本空间，概率在缩小的空间内重新分配。**

📐 **定义　条件概率（Conditional Probability）**：P(A|B) = P(A∩B) / P(B)。"在 B 发生的条件下 A 的概率" = "A 和 B 同时发生的概率"除以"B 发生的概率"。分母 P(B) 就是那个"缩小的样本空间"。

这个看起来简单的公式，是整个贝叶斯统计、朴素贝叶斯分类器、隐马尔可夫模型、以及现代概率图模型的理论基础。

💻 **代码　用模拟数据亲手计算条件概率**




In [ ]:
import numpy as np

np.random.seed(42)
n = 10000

# 模拟数据：人群中的两个特征
age = np.random.normal(35, 12, size=n)           # 年龄
income = np.random.normal(5000, 2000, size=n)    # 月收入
clicked = (age > 25) & (income > 4000)            # 点击行为（简化规则）
clicked = clicked.astype(int)

# P(点击) —— 无条件概率
p_click = clicked.mean()
print(f"P(点击) = {p_click:.3f}")

# P(点击 | 年龄>25) —— 条件概率
age_gt_25 = age > 25
p_click_given_age = clicked[age_gt_25].mean()
print(f"P(点击 | 年龄>25) = {p_click_given_age:.3f}")
print(f"  已知年龄>25 后，点击概率从 {p_click:.3f} 变为 {p_click_given_age:.3f}")

# P(点击 | 年龄>25 且 收入>6000)
both = (age > 25) & (income > 6000)
p_click_given_both = clicked[both].mean()
print(f"P(点击 | 年龄>25 且 收入>6000) = {p_click_given_both:.3f}")
print(f"\n规律: 条件越多，样本空间越小——概率在'剩下的世界'里重新分配")




> **关键洞察**：条件概率不是"改变现实"，而是"缩小你关注的范围"。每次获得新信息，你就在概率空间中画一个圈，只看圈内的比例。贝叶斯定理（16.2 节）把这个过程倒过来——从结果反推原因。

🔗 **AI 连接**：分类模型输出的 P(class | image) 本身就是一个条件概率——"给定这张图片，它属于猫类的概率"。整个监督学习的目标就是学习 P(label | input)。

---

### 16.2　贝叶斯定理 —— 从果推因

条件概率告诉你"已知原因，结果概率多少"。但现实中最常遇到的问题恰好相反：**你看到了结果，想知道原因的概率。** 医学诊断就是典型——患者检测呈阳性（结果），他真正患病的概率（原因）是多少？

贝叶斯定理给了这个反推的公式：

$$P(\text{患病} | \text{阳性}) = \frac{P(\text{阳性} | \text{患病}) \cdot P(\text{患病})}{P(\text{阳性})}$$

**经典反直觉案例**：某疾病发病率 1%，检测准确率 99%（患者 99% 测出阳性，健康人 99% 测出阴性）。你做了检测，结果是阳性。你真正患病的概率是多少？直觉告诉你是 99%——**错！真实答案约 50%。**

为什么？因为在 10000 人中，只有 100 人真正患病（1%），其中 99 人测出阳性（真阳性）。但 9900 个健康人中，也有 99 人测出阳性（1% 假阳性率）。所以所有阳性者中：99 真阳性 + 99 假阳性 = 198 人阳性，真阳性只占 99/198 ≈ 50%。

📐 **定义　贝叶斯定理（Bayes' Theorem）**：P(H|E) = P(E|H)·P(H) / P(E)。H 是假设（患病），E 是证据（阳性）。**先验概率 P(H) 乘上似然比 P(E|H)/P(E) 得到后验概率 P(H|E)。**

💻 **代码　贝叶斯更新：亲手验证"99% 准确率 → 50% 真实患病率"**




In [ ]:
# 贝叶斯定理计算器
def bayes_theorem(p_h, p_e_given_h, p_e_given_not_h):
    """
    P(H|E) = P(E|H) * P(H) / P(E)
    其中 P(E) = P(E|H)*P(H) + P(E|not H)*P(not H)
    """
    p_not_h = 1 - p_h
    p_e = p_e_given_h * p_h + p_e_given_not_h * p_not_h  # 全概率公式
    p_h_given_e = p_e_given_h * p_h / p_e
    return p_h_given_e

# 案例参数
prevalence = 0.01         # 发病率 1%
sensitivity = 0.99         # 真阳性率: P(阳性|患病) = 99%
false_positive = 0.01      # 假阳性率: P(阳性|健康) = 1%

p_sick_given_positive = bayes_theorem(
    p_h=prevalence,
    p_e_given_h=sensitivity,
    p_e_given_not_h=false_positive
)

print(f"检测准确率: {sensitivity*100:.0f}%")
print(f"疾病发病率: {prevalence*100:.0f}%")
print(f"\nP(患病 | 阳性) = {p_sick_given_positive*100:.1f}%")
print(f"——不是 99%，而是约 50%！")

# 模拟验证
np.random.seed(42)
n_people = 100000
is_sick = np.random.random(n_people) < prevalence

# 检测结果
test_result = np.zeros(n_people, dtype=bool)
test_result[is_sick] = np.random.random(is_sick.sum()) < sensitivity    # 患病者测出阳性
test_result[~is_sick] = np.random.random((~is_sick).sum()) < false_positive  # 健康者假阳性

positive_tests = test_result.sum()
true_positives = (test_result & is_sick).sum()
sim_result = true_positives / positive_tests
print(f"\n模拟验证 (n={n_people}):")
print(f"  阳性总人数: {positive_tests}")
print(f"  真阳性(患病且测出阳性): {true_positives}")
print(f"  P(患病|阳性) ≈ {sim_result*100:.1f}% ✓")




> **关键洞察**：贝叶斯定理的核心教训——**永远要考虑基础概率（先验）。** 罕见疾病的阳性结果多半是假阳性，不是因为检测不准，而是因为疾病本身太罕见。AI 中不平衡分类问题的"准确率陷阱"正是同一个数学原理——99% 的样本是负类，模型全部预测负类就获得 99% 的准确率，但对正类毫无识别能力。

🔗 **AI 连接**：贝叶斯定理是朴素贝叶斯分类器（16.3 节）的理论基础；也是变分自编码器（VAE）和贝叶斯神经网络中"先验→后验更新"的数学框架。

---

### 16.3　朴素贝叶斯分类器 —— 垃圾短信识别的数学原理

朴素贝叶斯（Naive Bayes）虽然名字里有"naive"（因为它做了一个非常天真但有效的假设：所有特征相互独立），但它是实践中极其成功的分类器——垃圾邮件过滤、情感分析、文档分类等任务中经常作为强基线。

**原理**：给定一条短信的词，计算 P(垃圾 | 词₁, 词₂, ..., 词ₙ) 和 P(正常 | 词₁, 词₂, ..., 词ₙ)，选概率大的那个。用贝叶斯定理展开后，"朴素"假设（特征独立）让 P(词₁, 词₂, ..., 词ₙ | 类别) = P(词₁ | 类别) × P(词₂ | 类别) × ... × P(词ₙ | 类别)，计算变得极为简单——只需要统计每个词在各类别中出现的频率。

💻 **代码　20 行实现朴素贝叶斯垃圾短信分类器**




In [ ]:
import numpy as np

# 模拟短信数据：每条短信是几个关键词
spam_msgs = [
    "免费 大奖 点击 链接", "恭喜 中奖 免费 领取",
    "优惠 限时 免费 抢购", "百万 大奖 点击 免费",
    "赚钱 机会 免费 点击",
]
ham_msgs = [
    "明天 开会 时间 下午", "晚上 一起 吃饭 吧",
    "会议 纪要 请 查收", "周末 有空 吗 约",
    "文件 已经 发 邮箱",
]

# 统计每个词在各类别中的出现次数（加 1 做拉普拉斯平滑）
from collections import Counter

def train(spam_list, ham_list):
    spam_words = ' '.join(spam_list).split()
    ham_words = ' '.join(ham_list).split()
    spam_total = len(spam_words)
    ham_total = len(ham_words)
    vocab = set(spam_words + ham_words)
    V = len(vocab)

    # P(word | class) with Laplace smoothing
    spam_probs = {w: (spam_words.count(w) + 1) / (spam_total + V) for w in vocab}
    ham_probs  = {w: (ham_words.count(w) + 1) / (ham_total + V) for w in vocab}
    return spam_probs, ham_probs, len(spam_list), len(ham_list)

spam_probs, ham_probs, n_spam, n_ham = train(spam_msgs, ham_msgs)
p_spam = n_spam / (n_spam + n_ham)

def predict(msg):
    """预测一条短信是否是垃圾"""
    words = msg.split()
    # log 避免大量小概率连乘导致下溢
    score_spam = np.log(p_spam)
    score_ham = np.log(1 - p_spam)
    for w in words:
        score_spam += np.log(spam_probs.get(w, 1e-6))
        score_ham += np.log(ham_probs.get(w, 1e-6))
    return "垃圾" if score_spam > score_ham else "正常"

# 测试
tests = [
    ("免费 大奖 点击 免费", "垃圾"),
    ("明天 开会 下午", "正常"),
    ("限时 优惠 赚钱", "垃圾"),
    ("一起 吃饭 周末", "正常"),
]
print(f"{'短信':<22} {'预测':<6} {'实际':<6} {'结果'}")
for msg, true_label in tests:
    pred = predict(msg)
    print(f"{msg:<22} {pred:<6} {true_label:<6} {'✓' if pred==true_label else '✗'}")




> **关键洞察**：朴素贝叶斯的"naive"假设（词之间独立）在现实中明显不成立——"免费"和"大奖"高度相关——但它仍然工作得非常好。原因：对于分类来说，只需要概率的相对大小正确，不需要绝对概率精确。**模型的简单性 + 数据充足 = 强基线。** 这个模式在 AI 中反复出现——有时最简单的模型反而是最实用的。

🔗 **AI 连接**：第 19 章交叉熵损失的推导中，最大化似然等价于最小化交叉熵——贝叶斯思维贯穿全书。第 27 章逻辑回归的 sigmoid 输出可以看作 P(类别=1|输入) 的贝叶斯后验估计。

---

### 16.4　Agent 的"自知之明" —— 用后验概率决定何时求助

朴素贝叶斯的输出不仅是"垃圾/正常"的硬分类——它还给出了每个类别的后验概率。这意味着 Agent 不仅知道"我认为这是垃圾"，还知道"我有多确定"。

这个"确定度"在 Agent 系统中至关重要。假设 Agent 判断"用户想订机票"的后验概率仅为 0.55（略高于 0.5），而判断"用户想退票"的概率为 0.45。如果 Agent 径直执行订票操作，45% 的概率做错——**与其强行猜测导致不可逆的错误，不如暂停并反问用户澄清意图。**

这就是 Agent 的"自知之明"：当最大后验概率低于某个置信度阈值（如 0.7）时，Agent 应触发澄清流程，而非盲目执行。

📐 **核心原则**：**P(action | input) < threshold → 暂停操作并反问用户。** 阈值的选择是一个业务决策——订机票（不可逆、代价高）用高阈值（0.8~0.9），推荐电影（可逆、代价低）用低阈值（0.5）。贝叶斯后验概率 = Agent 做出"自知之明"决策的数学依据。

💻 **代码　意图分类器 + 低置信度触发澄清**




In [ ]:
import numpy as np

np.random.seed(42)

# 模拟一个简单的意图分类器：5 种意图
intents = ["订机票", "查天气", "设提醒", "播放音乐", "退票"]
n_intents = len(intents)

# 模拟 10 条用户输入的分类概率（softmax 输出）
# 每行是一条输入在各意图上的概率分布
queries = [
    "帮我订一张明天去上海的机票",      # 应该高置信 → 订机票
    "今天天气怎么样",                   # 高置信 → 查天气
    "帮我订个什么东西来着",             # 低置信 → 需要澄清
    "那个票帮我处理一下",               # 低置信（订票 vs 退票不明确）
    "播放周杰伦的歌",                   # 高置信 → 播放音乐
    "提醒我下午三点开会",               # 高置信 → 设提醒
    "帮我把那个取消掉",                 # 低置信 → 取消什么？
    "明天上海天气如何",                 # 高置信 → 查天气
    "放点音乐",                        # 高置信 → 播放音乐
    "帮我把之前的那个弄一下",           # 极低置信 → 必须澄清
]

# 为每条 query 生成模拟的意图概率（使高/低置信度交替出现）
def simulate_probs(query, high_conf=True):
    """生成模拟概率分布。high_conf=True 时概率集中在单一意图。"""
    probs = np.random.dirichlet(np.ones(n_intents) * 0.5)  # 随机分布
    if high_conf:
        # 选一个意图大幅提升概率
        top = np.random.randint(0, n_intents)
        probs[top] += np.random.uniform(0.4, 0.6)
    probs = probs / probs.sum()
    return probs

# 模拟：奇数索引高置信，偶数索引低置信
threshold = 0.7
print(f"置信度阈值: {threshold}\n")
print(f"{'Query':<30} {'Top Intent':<10} {'Max Prob':<10} {'Action'}")
print("-" * 68)

for i, query in enumerate(queries):
    probs = simulate_probs(query, high_conf=(i % 2 == 0))
    max_idx = np.argmax(probs)
    max_prob = probs[max_idx]

    if max_prob >= threshold:
        action = f"✓ 执行: {intents[max_idx]}"
    else:
        # 找 top-2 意图生成澄清问题
        top2 = np.argsort(probs)[-2:][::-1]
        action = f"⚠ 澄清: 您是想{intents[top2[0]]}还是{intents[top2[1]]}?"

    print(f"{query:<30} {intents[max_idx]:<10} {max_prob:<10.3f} {action}")

high_conf_count = sum(1 for i in range(len(queries)) if i % 2 == 0)
print(f"\n高置信度 query: {high_conf_count} 条 → 直接执行")
print(f"低置信度 query: {len(queries) - high_conf_count} 条 → 触发澄清反问")




> **关键洞察**：16.2 节贝叶斯定理中"99% 准确检测 → 阳性后患病率仅 50%"的教训，在这里直接转化为 Agent 的设计原则——**永远不要信任单一高置信度的表面数字，永远考虑"如果我错了代价有多大"。** 这不仅是数学，更是产品安全：一个错误执行的退票操作，比十次正确执行带来的用户信任损失更大。让 Agent 在不确定时学会说"我不确定，你能否澄清一下？"——比盲目执行更聪明。

🔗 **AI 连接**：本节的意图分类器 + §5.5 的 RAG 检索 + §4.5 的状态迁移 + §14.7 的显存管理 = 一个完整的 Agent 推理循环。第 19.7 节将引入困惑度（PPL）作为另一个维度的"确定度"指标——PPL 极高意味着输入可能是越狱攻击。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）条件概率 P(A|B) 中的分母 P(B) 有什么几何含义？为什么它能"缩小样本空间"？
2. （概念）用贝叶斯定理解释：为什么"99% 准确的检测 + 1% 发病率 → 阳性后患病率仅 ~50%"？谁是真凶（假阳性太多，还是真阴性太少）？
3. （代码）扩展 16.3 节的垃圾短信分类器，增加更多训练样本（至少 10 条垃圾+10 条正常），测试 5 条新短信的准确率。加入拉普拉斯平滑的效果验证（不加平滑时对未见词的预测结果对比）。
4. （代码）模拟一个 4 意图分类器，生成 8 条 query 的概率分布（4 条高置信、4 条低置信）。设定阈值=0.7，对每条 query 输出"直接执行"或"触发澄清（列出 top-2 候选意图）"。统计触发澄清的比例。
---


> 🔗 **章末钩子**：概率分布描述"单一变量的随机性"，条件概率描述"变量之间的依赖关系"。但要量化"两个变量一起变"的程度——比如身高和体重同向变化的强度——你需要协方差。
> 预览：下一章——**期望、方差与协方差矩阵**，第二次知识融合。


> 💡 **提示**：完成后运行 Kernel -> Restart & Run All 验证所有代码块。
